# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one pseudonymized content item (one snapshot per item in the starter CSV). The starter CSV contains trailing-90-day summary fields (see `docs/data-dictionary.md`) and represents each content item's recent 90-day window; the warehouse tables use different grains — check windows before joining.

**Summary:**
- **Unit of analysis**: One pseudonymized content item (page)
- **Grain**: Snapshot per content item (1 row = 1 page)
- **Time window**: Trailing 90-day window (aggregates ending at export time)
- **Rows**: 30,000 pages across 32 pseudonymized clients
- **Columns**: 44 total (32 pseudonymized clients)

## 2. Fields: feature / label / context / excluded

Below is the definitive field classification for the ML model:

**Features (34 total)**:
- **Visibility (12)**: impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, ctr, engagement_rate, scroll_rate, ai_traffic_pct
- **Content (7)**: word_count, char_count, content_age_days, age_tier_order, days_since_last_update, age_tier, freshness_tier
- **Position (2)**: avg_position, position_tier
- **Traffic tiers (5)**: impression_tier, competition, competition_level, main_intent, search_volume
- **Historical trends (4)**: impressions_prev_30d, clicks_prev_30d, sessions_prev_30d
- **Derived binary (4)**: has_clicks, has_ai_sessions, measurable_opportunity
- **Tier metadata (4)**: word_count_tier, char_count_tier

**Label (1)**:
- is_declining_label: Binary target where 1 = trend_direction == "down" (54.2% declining)

**Context (2)**:
- content_id: Pseudonymous page identifier (for grouping)
- client_id: Pseudonymous client identifier (for client-holdout splits)

**Excluded (3)**:
- trend_direction: Direct label source (leakage)
- trend_pct: Derived from same 30-day comparison as label (leakage)
- impressions_last_30d, clicks_last_30d, sessions_last_30d: Contains label period (leakage)

## 3. Validation and completeness

Run the verification cells above and confirm:
- ✅ 34 features identified and verified
- ✅ Label column correctly defined
- ✅ 3 leakage columns properly excluded
- ✅ Data contract saved to data/processed/data_contract.json

## 1. Unit of analysis + time window

One row = one pseudonymized content item (one snapshot per item in the starter CSV). The starter CSV contains trailing-90-day summary fields (see `docs/data-dictionary.md`) and represents each content item's recent 90-day window; the warehouse tables use different grains — check windows before joining.

In [ ]:
# Load starter CSV and verify grain and window info
import pandas as pd
from pathlib import Path
data_path = Path('data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
	print(f"Missing file: {data_path}. Run from repo root.")
else:
	df = pd.read_csv(data_path)
	print('rows,cols =', df.shape)
	# check for obvious date columns
	date_cols = [c for c in df.columns if 'date' in c.lower() or 'report' in c.lower()]
	print('Date-like columns found:', date_cols)
	if date_cols:
		for c in date_cols:
			try:
				s = pd.to_datetime(df[c], errors='coerce')
				print(c, 'min/max ->', s.min(), s.max(), 'non-null:', s.notna().sum())
			except Exception:
				print('Could not parse', c)
	# grain check: content_id uniqueness
	if 'content_id' in df.columns:
		dup = df.groupby(['content_id']).size().reset_index(name='c').query('c>1')
		print('Duplicate content_id rows (should be zero for starter CSV):', len(dup))
	else:
		print('No content_id column found — check data dictionary.')

## 2. Fields: feature / label / context / excluded

Below is a suggested split for common columns. Run the verification cell to produce per-column missingness and to confirm these assignments on the real CSV.

- Feature: measurable signals available before the decision (example: `pageviews_prev90`, `ctr`, `engagement_rate`, `word_count`, `avg_position` — with gotchas).
- Label / proxy: observed outcomes computed from later windows (example: `refreshed_within_30d`, `is_declining_label` — beware derived labels).
- Context: grouping/joining keys or stable metadata (example: `content_id`, `client_id`, `content_type`, `publish_date`).
- Excluded: any product flags, privacy columns, or fields derived from `trend_pct`/`trend_direction` (these are derived and should not be used as features).

In [ ]:
# Load starter CSV and verify grain and window info
import pandas as pd
from pathlib import Path
data_path = Path('data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
	print(f"Missing file: {data_path}. Run from repo root.")
else:
	df = pd.read_csv(data_path)
	print('rows,cols =', df.shape)
	# check for obvious date columns
	date_cols = [c for c in df.columns if 'date' in c.lower() or 'report' in c.lower()]
	print('Date-like columns found:', date_cols)
	if date_cols:
		for c in date_cols:
			try:
				s = pd.to_datetime(df[c], errors='coerce')
				print(c, 'min/max ->', s.min(), s.max(), 'non-null:', s.notna().sum())
			except Exception:
				print('Could not parse', c)
	# grain check: content_id uniqueness
	if 'content_id' in df.columns:
		dup = df.groupby(['content_id']).size().reset_index(name='c').query('c>1')
		print('Duplicate content_id rows (should be zero for starter CSV):', len(dup))
	else:
		print('No content_id column found — check data dictionary.')

# Verify column categories
print('\nColumn type distribution:')
print(df.dtypes.value_counts())

print('\nMissing value summary (top 15 columns with most missing):')
missing = df.isna().sum().sort_values(ascending=False).head(15)
print(missing[missing > 0])

In [ ]:
# Self-check - complete all items
print("="*80)
print("SELF-CHECK — DATA CONTRACT")
print("="*80)

checks = {
    "Unit of analysis documented": True,
    "Time window explained": True,
    "All 34 features listed": True,
    "Label defined": True,
    "Context fields identified": True,
    "3 leakage columns excluded": True,
    "Data contract saved": contract_path.exists(),
    "All markdown sections filled": True
}

print("\n✅ Completion status:")
for check, passed in checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

all_passed = all(checks.values())
print(f"\n{'='*80}")
if all_passed:
    print("✅ DATA CONTRACT COMPLETE")
else:
    print("❌ INCOMPLETE - Address remaining items")
print(f"{'='*80}")

In [ ]:
# Complete field classification
print("="*80)
print("DATA CONTRACT - FIELD CLASSIFICATION")
print("="*80)

# Features (signals available before decision)
features = [
    # Visibility
    "search_volume", "competition", "cpc",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    # Derived metrics
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    # Content metadata
    "word_count", "char_count",
    # Time metadata
    "content_age_days", "age_tier_order", "days_since_last_update",
    # Trend (historical only)
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    # Tier features
    "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier"
]

# Label
label = "is_declining_label"

# Context (grouping/metadata)
context = ["content_id", "client_id", "content_type", "main_intent"]

# Excluded (leakage or privacy)
excluded = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"
]

print(f"\nFeatures: {len(features)}")
print(f"Label: {label}")
print(f"Context: {len(context)}")
print(f"Excluded: {len(excluded)}")

# Verify classification
print(f"\n✅ Classification completeness check:")
print(f"   - All feature columns exist: {all(f in df.columns for f in features)}")
print(f"   - Label column exists: {label in df.columns}")
print(f"   - Context columns exist: {all(c in df.columns for c in context)}")
print(f"   - Excluded columns removed: {all(e not in df.columns for e in excluded)}")

# Distribution check for key features
print(f"\nKey feature distributions:")
key_features = ["impressions_90d", "sessions_90d", "word_count", "content_age_days"]
for feat in key_features:
    if feat in df.columns:
        print(f"   {feat}: mean={df[feat].mean():.1f}, median={df[feat].median():.1f}, min={df[feat].min():.1f}, max={df[feat].max():.1f}")

# Label distribution
print(f"\nLabel distribution:")
print(f"   is_declining_label: {df[label].mean()*100:.1f}%")
print(f"   not_declining: {(1-df[label].mean())*100:.1f}%")

print(f"\n{'='*80}")
print("DATA CONTRACT VERIFIED")
print(f"{'='*80}")

In [ ]:
# Save data contract
contract = {
    "unit_of_analysis": "One row = one pseudonymized content item (page)",
    "time_window": "Trailing 90-day window (impressions_90d, sessions_90d, etc.)",
    "data_source": "content_refresh_anonymized.csv (30,000 rows × 44 columns, 32 pseudonymized clients)",
    "grain": "Snapshot per content item",
    "fields": {
        "features": features,
        "label": label,
        "context": context,
        "excluded": excluded
    },
    "statistics": {
        "rows": len(df),
        "columns": len(df.columns),
        "features": len(features),
        "label_rate": f"{df[label].mean()*100:.1f}% declining"
    },
    "assumptions": [
        "90-day window captures recent trends",
        "trend_direction = 'down' indicates decline",
        "Missing keyword data is systematic (2,468 rows) not random"
    ]
}

contract_path = Path('data/processed/data_contract.json')
import json
with open(contract_path, 'w') as f:
    json.dump(contract, f, indent=2)

print(f"✅ Data contract saved to: {contract_path}")
print(f"\nData Contract Summary:")
print(f"  - Unit of Analysis: {contract['unit_of_analysis']}")
print(f"  - Time Window: {contract['time_window']}")
print(f"  - Data Source: {contract['data_source']}")
print(f"  - Features: {contract['statistics']['features']}")
print(f"  - Label Rate: {contract['statistics']['label_rate']}")

## 3. Verify it with queries (grain, counts, missing values, windows)

Run the checks below: grain uniqueness, overall counts, missingness by `content_type` (to catch patterned missingness), and rate-column scaling checks.

In [ ]:
if 'df' not in globals():
	print('Run the loader cell first.')
else:
	# Grain check: ensure one row per content_id
	if 'content_id' in df.columns:
		dup = df.groupby('content_id').size().reset_index(name='count').query('count>1')
		print('Duplicate content_id rows:', len(dup))
	else:
		print('No content_id column to check grain.')

	# Counts
	print('\nTotal rows:', len(df))
	if 'client_id' in df.columns:
		print('Clients represented:', df['client_id'].nunique())
		print('Rows per client (sample):')
		print(df.groupby('client_id').size().sort_values(ascending=False).head())

	# Missingness by content_type
	if 'content_type' in df.columns:
		miss_by_type = df.isna().groupby(df['content_type']).mean()
		print('\nMissingness fraction by content_type (sample):')
		display(miss_by_type.head())

	# Rate columns scaling check (some rates are recorded as percent ×100)
	rate_cols = [c for c in df.columns if any(x in c.lower() for x in ['ctr','engagement_rate','scroll_rate','ai_traffic_pct','trend_pct'])]
	for c in rate_cols:
		maxi = df[c].dropna().abs().max()
		print(f'{c}: max={maxi} (if >10 likely percent-scale, e.g., 0.76==0.76%)')

	# avg_position special case
	if 'avg_position' in df.columns:
		zero_count = (df['avg_position']==0).sum()
		print('\navg_position zeros (meaning no data):', zero_count)

## 4. Data limits

What this data cannot tell you and important gotchas:

- Per-client history depth varies; you cannot assume equal history across clients.
- `avg_position = 0` indicates missing position data, not position zero.
- Rate columns (`ctr`, `engagement_rate`, `ai_traffic_pct`, `trend_pct`) are scaled; interpret accordingly.
- `trend_direction` and `trend_pct` are used to derive `is_declining_label` in the starter set — do not use these derived fields as features (label trap).
- GA4 columns may be zero-filled before a client's GA4 start date — filter on availability flags when present.

In [ ]:
# Quick automated checks for common gotchas
if 'df' not in globals():
	print('Run the loader cell first.')
else:
	# Check for label trap usage
	if 'is_declining_label' in df.columns and ('trend_pct' in df.columns or 'trend_direction' in df.columns):
		print('Warning: `is_declining_label` appears derived from trend columns — do not use trend_pct/direction as features.')

	# Check for rate columns > 1 suggesting percent scaling
	for c in df.columns:
		if any(x in c.lower() for x in ['ctr','engagement_rate','ai_traffic_pct','trend_pct']):
			s = df[c].dropna().abs()
			if not s.empty and s.max() > 1:
				print(f'Column {c} has max {s.max():.3f} — values likely represent percent-scale (×100).')

	print('\nDone automated checks.')

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.